# 프로젝트: Seq2seq으로 번역기 만들기 (복습)

아이펠 DeepDive(NLP) — **Seq2seq 기반 번역기** 프로젝트를 Mac 로컬 환경에서 복습합니다.

| 강 | 내용 |
|---|---|
| 1 | 프로젝트 소개 & 준비물 |
| 2 | 데이터 전처리 (영어-스페인어 말뭉치, SentencePiece) |
| 3 | 모델 설계 (GRU Encoder-Decoder + **Bahdanau Attention**) |
| 4 | 훈련하기 (Optimizer & Loss → train_step → 훈련/평가/Attention Map) |
| 5 | **[프로젝트]** Seq2Seq으로 한국어-영어 번역기 만들기 |

**아이펠 서버 환경과 다른 점 (Mac 로컬용으로 수정됨)**

- 경로: `~/work/s2s_translation` → `~/aiffel-review/s2s_translation`
- 장치: `cuda` → **`mps`** (Apple Silicon GPU)
- 한글 폰트: 나눔폰트 apt 설치 → Mac 내장 **AppleGothic**
- 한국어 형태소 분석: KoNLPy mecab → **python-mecab-ko** (동일한 mecab 엔진)

> 커널은 **Python (aiffel-review)** 를 선택하세요. 필요한 패키지(torch, sentencepiece,
> pandas, scikit-learn, tqdm, matplotlib, python-mecab-ko)는 이미 설치되어 있습니다.

---
## 1. Seq2seq으로 번역기 만들기

이전 시간에 배운 **Sequence-to-Sequence** 구조 — 두 개의 RNN 모듈을 **Encoder-Decoder**
구조로 결합한 모델 — 를 직접 만들어보며 이해하는 프로젝트입니다.
여기에 **Attention 기법**을 추가하여 성능도 높여봅니다.

- 실습: **영어 → 스페인어** 말뭉치 사용 (2~4강)
- 프로젝트: **한국어 → 영어** 말뭉치 사용 (5강)

### 학습 목표
- 데이터를 필요에 맞게 전처리할 수 있다.
- Encoder-Decoder 구조를 이해할 수 있다.
- Encoder-Decoder 구조를 다른 사람에게 설명할 수 있다.

### 준비물 — 프로젝트 디렉토리 생성

원문은 터미널에서 `mkdir -p ~/work/s2s_translation` 을 실행하지만,
여기서는 노트북 안에서 Mac 경로에 바로 만듭니다.

In [ ]:
import os

# 프로젝트 루트 디렉토리 (아이펠의 ~/work/s2s_translation 에 해당)
PROJECT_DIR = os.path.expanduser("~/aiffel-review/s2s_translation")
os.makedirs(PROJECT_DIR, exist_ok=True)

# 이후 상대 경로로 저장되는 파일들(말뭉치, spm 모델)이 모두 이 폴더에 모이도록
# 작업 디렉토리 자체를 옮겨줍니다.
os.chdir(PROJECT_DIR)
print("작업 디렉토리:", os.getcwd())

### 준비물 — matplotlib 한글 폰트

실습 말미에 **Attention Map**을 시각화할 때 한국어 라벨이 등장합니다.
matplotlib 기본 폰트는 한국어를 지원하지 않아서 폰트를 바꿔줘야 해요.

- 아이펠 서버(리눅스): `apt-get install fonts-nanum` 후 나눔바른고딕 지정
- **Mac: 내장 폰트인 `AppleGothic`을 그대로 쓰면 됩니다** (설치 불필요)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import logging

# font_manager가 뿜는 불필요한 경고 로그를 꺼줍니다.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# Mac 내장 한글 폰트 사용 (아이펠 서버에서는 NanumBarunGothic.ttf 경로를 지정)
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호 깨짐 방지

print(f"설정된 폰트: {plt.rcParams['font.family']}")

---
## 2. 데이터 전처리

### 데이터 준비하기

먼저 프로젝트에 사용될 라이브러리를 `import` 합니다.

In [ ]:
import os
import re
import urllib.request
import zipfile
import tarfile
import sentencepiece as spm
import pandas as pd

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm
import random

# 원문: torch.device("cuda" if ...) — Mac에서는 Apple Silicon GPU인 mps를 사용합니다.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(torch.__version__)
print("사용 장치:", device)

데이터 다운로드에는 `urllib.request.urlretrieve()` 함수를 사용합니다.
URL로부터 zip 파일을 내려받고, `zip_ref.extractall()` 함수로 폴더 안 압축 파일을
찾아 해제까지 해주는 코드입니다. 아래 소스를 실행해 데이터를 다운로드받아 주세요.

In [ ]:
# 데이터셋 저장 폴더: ~/aiffel-review/s2s_translation/datasets
dataset_dir = os.path.join(PROJECT_DIR, "datasets")
os.makedirs(dataset_dir, exist_ok=True)

zip_path = os.path.join(dataset_dir, "spa-eng.zip")

if not os.path.exists(zip_path):
    print("데이터 다운로드 중...")
    url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    urllib.request.urlretrieve(url, zip_path)
    print("다운로드 완료!")

data_folder = os.path.join(dataset_dir, "spa-eng")
if not os.path.exists(data_folder):
    print("압축 해제 중...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("압축 해제 완료!")

path_to_file = os.path.join(data_folder, "spa.txt")

print("데이터셋 디렉토리:", os.listdir(dataset_dir))

다운로드받은 데이터를 읽어온 후, 형태를 확인해보죠!

In [ ]:
df = pd.read_csv(path_to_file, sep="\t", names=["eng", "spa"])
df.head()

### 데이터 전처리: 정제하기

데이터는 `\t` 기호를 기준으로 영어와 스페인어가 병렬 쌍을 이루고 있습니다.
`\t` 를 구분자로 이미 분리했으니, 이제 **특수문자 노이즈를 정제**할 차례입니다.

- 구두점(`?.!,`) 앞뒤에 공백을 넣어 단어와 분리
- 연속 공백은 하나로
- 영문자와 구두점 이외의 문자(스페인어의 역물음표 `¿` 등)는 모두 제거

> 사실 스페인어에서는 역 물음표(¿)와 역 느낌표(¡)를 일반적으로 사용하지만,
> 이해를 돕기 위해 이상한 기호 취급하여 삭제합니다. 양해 바랍니다!

In [ ]:
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()          # 소문자화 + 양끝 공백 제거

    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)   # 구두점 앞뒤에 공백 삽입
    sentence = re.sub(r'[" "]+', " ", sentence)          # 연속 공백 → 하나로
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)  # 영문/구두점 외 문자 제거

    sentence = sentence.strip()

    return sentence

print("슝~")

전처리 과정에서 원래는 문장의 시작 문자 `<start>`, 종료 문자 `<end>` 를 붙여줍니다.
Encoder 입력에는 굳이 필요 없지만, **Decoder의 입력 문장과 라벨로 쓸 출력 문장에는 꼭
필요**합니다 — Decoder는 첫 입력으로 쓸 시작 토큰과 문장 생성 종료를 알리는 끝 토큰이
반드시 필요하기 때문이죠. (여기서는 뒤에서 SentencePiece의 `bos/eos` id로 붙입니다.)

원활한 학습을 위해 **데이터는 상위 3만 개만 사용**하도록 하겠습니다.

In [ ]:
df = df[:30000]

df["eng"] = df["eng"].apply(preprocess_sentence)
df["spa"] = df["spa"].apply(lambda x: preprocess_sentence(x))  # apply에 lambda를 쓰는 예

df.head()

### 데이터 전처리: 토큰화

`SentencePiece` 를 사용하려면 텍스트 파일이 필요합니다.
먼저 언어별 말뭉치를 텍스트 파일로 저장한 뒤, `SentencePiece` 를 활용해
Tokenizer를 생성하고 데이터를 변환하겠습니다.
**변환된 텐서는 80%의 훈련 데이터와 20%의 검증 데이터로 분리합니다!**

In [ ]:
df["eng"].to_csv("eng_corpus.txt", index=False, header=False, sep="\n", encoding="utf-8")
df["spa"].to_csv("spa_corpus.txt", index=False, header=False, sep="\n", encoding="utf-8")

print("파일 저장 완료: eng_corpus.txt, spa_corpus.txt")

In [ ]:
vocab_size = 3000
pad_id = 0   # 패딩 토큰
bos_id = 1   # 문장 시작(begin of sentence) = <start>
eos_id = 2   # 문장 종료(end of sentence)   = <end>
unk_id = 3   # 사전에 없는 토큰(unknown)

# Encoder(영어)용 tokenizer 학습
spm.SentencePieceTrainer.train(
    input="eng_corpus.txt",
    model_prefix="encoder_spm",
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id
)

# Decoder(스페인어)용 tokenizer 학습
spm.SentencePieceTrainer.train(
    input="spa_corpus.txt",
    model_prefix="decoder_spm",
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id
)

In [ ]:
encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load("encoder_spm.model")

decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load("decoder_spm.model")

토큰화가 어떻게 동작하는지 샘플 문장 하나로 확인해봅시다.

In [ ]:
eng_sample = df["eng"][10000]
spa_sample = df["spa"][10000]
print(eng_sample)
print(spa_sample)

문장을 인코딩한 뒤 START_TOKEN과 END_TOKEN을 붙여주겠습니다.

In [ ]:
enc_token = encoder_tokenizer.encode(eng_sample)
enc_token = [encoder_tokenizer.bos_id()] + enc_token + [encoder_tokenizer.eos_id()]
enc_token

위 토큰을 디코딩하면 어떻게 될까요?

In [ ]:
enc_decoding = encoder_tokenizer.decode(enc_token)
enc_decoding

이처럼 SentencePiece는 **디코딩 과정에서 START_TOKEN과 END_TOKEN 부분을 제외하고**
디코딩을 합니다.

### 데이터 전처리: Dataset 클래스 (패딩 포함)

START_TOKEN과 END_TOKEN은 데이터셋을 생성하는 과정에서 추가해줍니다.
일반적으로 Padding 작업은 `torch.nn.utils.rnn.pad_sequence` 같은 라이브러리를
사용하지만, 여기서는 과정을 좀 더 풀어서 직접 구현합니다.

- `src_ids` : Encoder 입력 (영어)
- `trg_input` : Decoder 입력 — `<bos> + 문장`
- `trg_label` : Decoder 정답 라벨 — `문장 + <eos>` (한 스텝 밀린 형태)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, data, encoder_tokenizer, decoder_tokenizer, max_len):
        self.data = data
        self.encoder_tokenizer = encoder_tokenizer
        self.decoder_tokenizer = decoder_tokenizer
        self.max_len = max_len
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['eng']
        trg_text = self.data.iloc[idx]['spa']

        src_ids = self.encoder_tokenizer.encode(src_text)
        trg_ids = self.decoder_tokenizer.encode(trg_text)

        src_ids = src_ids[:self.max_len]

        # Decoder의 입력에는 START_TOKEN과 END_TOKEN을 추가해줍니다. 단, 최대 길이 제한을 적용시킵니다.
        trg_input = [self.bos_id] + trg_ids[:self.max_len - 2] + [self.eos_id]
        trg_label = trg_ids[:self.max_len - 1] + [self.eos_id]

        # 길이가 짧은 경우 PAD_TOKEN을 추가해줍니다.
        src_ids = src_ids + [self.pad_id] * (self.max_len - len(src_ids))
        trg_input = trg_input + [self.pad_id] * (self.max_len - len(trg_input))
        trg_label = trg_label + [self.pad_id] * (self.max_len - len(trg_label))

        return torch.tensor(src_ids), torch.tensor(trg_input), torch.tensor(trg_label)

In [ ]:
train_ratio = 0.8  # 전체 길이의 80%
MAX_LEN = 30       # 임의의 값
BATCH_SIZE = 64

train_data = df.sample(frac=train_ratio, random_state=42)  # 80% 훈련 데이터
valid_data = df.drop(train_data.index)                     # 나머지 20%는 검증 데이터

train_data.reset_index(drop=True, inplace=True)
valid_data.reset_index(drop=True, inplace=True)

train_data = TranslationDataset(train_data, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)
validation_data = TranslationDataset(valid_data, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False)

DataLoader가 내놓는 배치의 shape을 확인해봅시다. 셋 다 `(BATCH_SIZE, MAX_LEN)` 이면 정상입니다.

In [ ]:
for src, trg_input, trg_label in train_loader:
    print(src.shape, trg_input.shape, trg_label.shape)
    break

---
## 3. 모델 설계

이제 **각각 1개의 GRU를 갖는 Encoder-Decoder 구조**를 설계합니다.

```
[Encoder]                        [Decoder]
call me .                        <start> Llamame .
   ↓ Embedding Layer                ↓ Embedding Layer
   ↓ (batch x length x embedding)   ↓ (batch x length x embedding)
  GRU  ──────── Encoder Output ──→ Attention + GRU
   ↓ (batch x length x units)       ↓
                                  Linear (fc_out)
                                    ↓ (batch x length x target vocab size)
                                  Llamame . <end>
```

- **Encoder는 모든 Time-Step의 Hidden State를 출력**으로 갖고,
- **Decoder는 Encoder의 출력과 Decoder의 t-1 Step의 Hidden State로 Attention을 취하여
  t Step의 Hidden State를 만들어 냅니다.**
- Decoder에서 t Step의 단어로 예측된 것과 실제 정답을 대조해 Loss를 구하고,
  생성된 t Step의 Hidden State는 t+1 Step의 Hidden State를 만들기 위해 다시 Decoder에 전달됩니다.
- **'t=1일 때의 Hidden State는 어떻게 정의할 것인가?'** — 일반적으로 **Encoder의 Final State**를
  Hidden State로 사용합니다.

Attention은 **Bahdanau** 방식을 사용합니다.

> 💡 **Bahdanau Attention이란?** Decoder의 직전 hidden state(query)와 Encoder의
> 모든 출력(key)을 작은 신경망(W1, W2, v)에 통과시켜 **덧셈(additive) 방식**으로
> 유사도(energy)를 구한 뒤 softmax로 가중치를 만드는 기법입니다.
> 소스 문장의 어느 단어에 집중할지를 매 스텝 동적으로 결정하게 해줍니다.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.W1 = nn.Linear(hidden_dim, hidden_dim)          # encoder_outputs 변환용
        self.W2 = nn.Linear(hidden_dim, hidden_dim)          # decoder hidden 변환용
        self.v = nn.Linear(hidden_dim, 1, bias=False)        # energy를 스칼라로 축소

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch_size, hidden_dim)
        # encoder_outputs: (src_len, batch_size, hidden_dim)

        src_len = encoder_outputs.shape[0]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)   # (batch_size, src_len, hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)   # (batch_size, src_len, hidden_dim)

        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))  # (batch_size, src_len, hidden_dim)
        attention = self.v(energy).squeeze(2)                # (batch_size, src_len)

        return nn.functional.softmax(attention, dim=1)       # (batch_size, src_len)

위 그림과 동일한 구조를 갖는 `Encoder` 클래스와 `Decoder` 클래스를 설계합니다.

**Encoder** — Embedding을 거쳐 GRU에 통과시키고, 모든 스텝의 출력(`outputs`)과
마지막 hidden state(`hidden`)를 반환합니다.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)

    def forward(self, src):
        # src : (src_len, batch_size)
        embedded = self.embedding(src)            # embedded : (src_len, batch_size, emb_dim)
        outputs, hidden = self.rnn(embedded)      # outputs : (src_len, batch_size, hidden_dim)

        return outputs, hidden

**Decoder** — 단어 1개(1 time-step)를 입력받아,

1. 직전 hidden state와 Encoder 출력으로 **attention 분포** `a`를 계산하고
2. `a`를 가중치로 Encoder 출력을 가중합해 **context vector**를 만들고
3. GRU를 한 스텝 진행한 뒤, **hidden state와 context vector를 결합(concat)**하여
   다음 단어를 예측합니다.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super(Decoder, self).__init__()

        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # Decoder RNN에는 embedding만 입력
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        # 출력층에는 hidden state와 attention value가 결합되어 입력
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        # input : (batch_size,)
        # hidden : (batch_size, hidden_dim)
        # encoder_outputs : (src_len, batch_size, hidden_dim)
        input = input.unsqueeze(0)                 # input : (1, batch_size)
        embedded = self.embedding(input)           # embedded : (1, batch_size, emb_dim)

        # attention distribution을 계산합니다. decoder의 이전 hidden state, s_{t-1}와 encoder의 출력 사용
        a = self.attention(hidden[-1], encoder_outputs)  # a : (batch_size, src_len)

        # H에 가중치를 부여해 attention value(Context vector) 계산
        a = a.unsqueeze(1)                                    # a : (batch_size, 1, src_len)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)    # encoder_outputs : (batch_size, src_len, hidden_dim)
        context = torch.bmm(a, encoder_outputs)               # context : (batch_size, 1, hidden_dim)
        context = context.permute(1, 0, 2)                    # context : (1, batch_size, hidden_dim)

        output, hidden = self.rnn(embedded, hidden)

        # 출력층에서는 현재 hidden state와 context vector를 결합하여 예측값 생성
        output = output.squeeze(0)                 # output : (batch_size, hidden_dim)
        context = context.squeeze(0)               # context : (batch_size, hidden_dim)
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, output_dim)

        return prediction, hidden, a.squeeze(1)

**Seq2SeqAttention** — Encoder와 Decoder를 하나로 묶는 최상위 모델입니다.

- **학습 모드** (`trg` 제공): 매 스텝 **정답 단어를 다음 입력으로 사용**합니다 (Teacher Forcing).
- **추론 모드** (`trg=None`): `<start>` 토큰부터 시작해 **자신이 예측한 단어**를 다음 입력으로
  사용하며, 모든 배치가 `<end>`에 도달하면 조기 종료합니다.

In [ ]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=30, bos_id=1, eos_id=2):
        # 학습 모드에서는 trg_len 사용, 추론 모드에서는 max_len까지 동적 생성
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        # 조기 종료를 위해 tensor가 아닌 리스트 사용
        outputs = []

        # 시각화를 위해 attention 저장
        attentions = []

        # 인코더를 통해 context 생성
        encoder_outputs, hidden = self.encoder(src)

        if trg is not None:
            for t in range(0, trg.shape[0]):
                input = trg[t]
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))

        else:
            # inference에서는 target(정답)이 없기 때문에 sos_token을 생성해줍니다.
            input = torch.full((batch_size,), bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            for t in range(max_len):
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))
                top1 = output.argmax(1)
                input = top1

                # 조기 종료 조건
                finished |= (top1 == eos_id)
                if finished.all():
                    break

        outputs = torch.cat(outputs, dim=0)        # (trg_len, batch_size, output_dim)
        attentions = torch.cat(attentions, dim=0)  # (trg_len, batch_size, src_len)

        return outputs, attentions

모델을 생성합니다. 임베딩 차원 256, hidden 차원 512를 사용합니다.

In [ ]:
# 원문: device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Mac에서는 위에서 만든 mps device를 그대로 사용합니다.

input_dim = len(encoder_tokenizer)    # 소스(영어) 단어 사전 크기 = 3000
output_dim = len(decoder_tokenizer)   # 타겟(스페인어) 단어 사전 크기 = 3000
emb_dim = 256
hid_dim = 512

In [ ]:
encoder = Encoder(input_dim, emb_dim, hid_dim).to(device)
attention = BahdanauAttention(hid_dim).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, attention).to(device)
model = Seq2SeqAttention(encoder, decoder, device).to(device)

In [ ]:
print(model)

코드를 실행하면 다음과 같은 형식의 결과가 나와야 합니다.

```
Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(3000, 256)
    (rnn): GRU(256, 512)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=512, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(3000, 256)
    (rnn): GRU(256, 512)
    (fc_out): Linear(in_features=1024, out_features=3000, bias=True)
  )
)
```

---
## 4. 훈련하기

지금까지는 Optimizer 안에 파라미터를 간편하게 입력했지만, Encoder-Decoder 구조의 경우
모델이 분리되어 파라미터를 각각 정의해줘야 합니다. 이번 코스에선 직접 구현을 하기보단
**구현체를 먼저 보고 이해하는 방향**으로 공부합니다!

### (1) Optimizer & Loss

- **Optimizer**: 모델이 학습할 때에 정답을 찾아가는 방법. 일반적으로 **Adam** 외의 것을
  사용하지 않으니 우선 Adam으로!
- **`nn.CrossEntropyLoss()`**: 모델이 출력한 **확률 분포**와 (One-hot이 아닌) **정수 인덱스
  답안**을 비교해 Cross Entropy 값을 구해줍니다.
  (`nn.NLLLoss()`라면 `[0.1, 0.2, 0.7]` 과 One-hot 라벨 `[0, 0, 1]` 을 비교,
  `nn.CrossEntropyLoss()`는 `[0.1, 0.2, 0.7]` 과 정수 인덱스 답안 `2` 를 비교)

**Q. 패딩(Padding) 과정을 진행하는 이유가 뭘까요?**

> 배치 안 문장들의 길이가 제각각이면 하나의 텐서로 묶을 수 없기 때문에,
> `MAX_LEN`에 맞춰 `<PAD>` 토큰으로 길이를 통일해주는 것입니다.

단, 모델에게 `<PAD>` 토큰이 **패딩을 위한 토큰이라고 명시하지 않으면** 모델은 데이터의
굉장히 많은 부분이 `<PAD>` 로 이뤄져 있다고 생각하게 됩니다. 쉽게 말해 **유난히 같은 답이
많은 객관식 시험** — 모델이 `<PAD>` 토큰만을 생성할 확률이 굉장히 높아집니다.

이를 방지하기 위해 **mask**가 사용됩니다. mask는 정답지에서 `<PAD>` 토큰을 찾아내어
그 부분에 대한 Loss는 구하지 않도록 하는 역할을 하죠. `CrossEntropyLoss(ignore_index=...)`
에 `<PAD>` 토큰의 인덱스(우리는 0)를 전달하면 됩니다.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)   # <PAD>(0) 위치는 Loss 계산에서 제외(mask)

print("슝~")

### (2) train_step 구현하기

`train_step()` 은 학습에 필요한 것을 모두 가져가 **Loss를 계산한 후 반환하는 함수**입니다.

학습 과정은 아래와 같습니다.

1. `model.train()`을 호출하여 모델을 학습 모드로 전환합니다.
2. `optimizer.zero_grad()`를 통해 이전 배치에서 계산된 기울기를 초기화합니다.
3. Encoder에 소스 문장을 전달해 **컨텍스트 벡터인 `enc_out` 을 생성**
4. **t=0일 때, Decoder의 Hidden State는 Encoder의 Final State로 정의.**
5. `<start>` 문장과 enc_out, Hidden State를 기반으로 **다음 단어(t=1)를 예측.**
6. 예측된 단어와 정답 간의 Loss를 구한 후, t=1의 **정답 단어를 다음 입력으로 사용** (예측 단어 사용X)
7. 반복!

In [ ]:
def train_step(model, data_loader, optimizer, criterion, epoch):
    model.train()
    epoch_loss = 0

    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}", leave=True)

    for src, trg_input, trg_label in progress_bar:
        # 모델의 입력 순서에 맞게 transpose 변환: (batch, len) → (len, batch)
        src = src.permute(1, 0).to(device)
        trg_input = trg_input.permute(1, 0).to(device)
        trg_label = trg_label.permute(1, 0).to(device)
        optimizer.zero_grad()

        outputs, _ = model(src, trg_input)

        # (trg_len, batch_size, output_dim)을 (batch_size * trg_len, output_dim)으로 변환
        outputs = outputs.reshape(-1, outputs.shape[-1])
        trg_label = trg_label.reshape(-1)

        loss = criterion(outputs, trg_label)
        loss.backward()

        # 기울기 폭발(gradient explosion) 방지를 위한 클리핑
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        optimizer.step()

        epoch_loss += loss.item()

        progress_bar.set_postfix(loss=loss.item())

    return epoch_loss / len(data_loader)

print("슝~")

### eval_step 정의하기 (실습)

Step 1에서 분리한 **Validation Set을 사용하는 `eval_step()`** 함수입니다.
train_step()을 마친 후, 곧이어 eval_step()을 진행하도록 합니다.

**주의! Evaluation 중에는 모델이 학습을 해서는 안 됩니다!**
→ `model.eval()`로 평가 모드 전환 + `torch.no_grad()`로 기울기 계산을 꺼줍니다.
(당연히 `loss.backward()`, `optimizer.step()`도 없습니다.)

In [ ]:
def eval_step(model, data_loader, optimizer, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():   # 평가 중에는 기울기를 계산하지 않습니다 (학습 방지)
        for src, trg_input, trg_label in data_loader:
            # 모델의 입력 순서에 맞게 transpose 변환
            src = src.permute(1, 0).to(device)
            trg_input = trg_input.permute(1, 0).to(device)
            trg_label = trg_label.permute(1, 0).to(device)

            outputs, _ = model(src, trg_input)

            # (trg_len, batch_size, output_dim)을 (batch_size * trg_len, output_dim)으로 변환
            outputs = outputs.reshape(-1, outputs.shape[-1])
            trg_label = trg_label.reshape(-1)

            loss = criterion(outputs, trg_label)

            total_loss += loss.item()

    return total_loss / len(data_loader)

print("슝~")

### (3) 훈련 시작하기

본격적으로 훈련을 시작합니다. 원문 기준 훈련 시간은 약 15분 정도
(아이펠 GPU 서버, 10 epochs). **Mac(mps)에서는 20 epochs에 더 오래 걸릴 수 있으니**,
시간이 부족하면 `EPOCHS`를 5~10으로 줄여서 실행해보세요.

`tqdm` 은 훈련의 진행 과정을 한눈에 볼 수 있게 해주는 라이브러리입니다.
매 epoch마다 `train_step()` → `eval_step()` 순서로 진행하며 두 Loss를 함께 출력합니다.

In [ ]:
%%time

EPOCHS = 20   # 시간이 부족하면 5~10으로 줄여보세요

for epoch in range(EPOCHS):
    train_loss = train_step(model, train_loader, optimizer, criterion, epoch)
    valid_loss = eval_step(model, validation_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}')

출력 예시 (원문):

```
Epoch  1: 100%|████| 375/375 [00:50<00:00,  7.45it/s, Loss 1.7839]
Test Epoch  1: 100%|████| 94/94 [00:09<00:00,  9.75it/s, Test Loss 1.4220]
Epoch  2: 100%|████| 375/375 [00:31<00:00, 11.74it/s, Loss 1.2333]
Test Epoch  2: 100%|████| 94/94 [00:02<00:00, 46.58it/s, Test Loss 1.1305]
...
```

Train Loss와 함께 Validation Loss도 꾸준히 내려가는지 지켜보세요.
(Validation Loss만 올라가기 시작하면 과적합 신호입니다.)

### 번역 성능 평가 & Attention Map 시각화

훈련이 완료된 모델은 아래 소스를 실행해 번역 성능을 평가할 수 있어요.
**Attention Map을 시각화하는 것은 보너스!**

`evaluate()` — 문장 하나를 전처리→인코딩→패딩한 뒤, 추론 모드(`trg=None`)로
모델을 돌려 번역 결과와 attention 행렬을 반환합니다.

In [ ]:
def evaluate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len=30):
    model.eval()

    sentence = preprocess_sentence(sentence)
    src_ids = encoder_tokenizer.encode(sentence)
    src_ids = src_ids[:max_len]
    src_ids = src_ids + [0] * (max_len - len(src_ids))   # 패딩 추가
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)   # (src_len, 1)

    with torch.no_grad():
        outputs, attentions = model(src_tensor, max_len=max_len)

    result = [decoder_tokenizer.decode([token.item()]) for token in outputs.argmax(2).squeeze(1)]

    if "<end>" in result:
        result = result[:result.index("<end>")]

    return result, sentence, attentions.squeeze(1).cpu().numpy()

In [ ]:
def plot_attention(attention, sentence, predicted_sentence):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.matshow(attention, cmap='viridis')   # 밝을수록 그 소스 단어에 강하게 집중했다는 뜻

    fontdict = {'fontsize': 14}

    ax.set_xticks(range(len(sentence)))
    ax.set_xticklabels(sentence, fontdict=fontdict, rotation=90)

    ax.set_yticks(range(len(predicted_sentence)))
    ax.set_yticklabels(predicted_sentence, fontdict=fontdict)

    plt.show()

In [ ]:
def translate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len=30):
    result, sentence, attention = evaluate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len)

    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

    # Attention 크기 조정 (trg_len, src_len)
    attention = attention[:len(result), :len(sentence.split())]

    plot_attention(attention, sentence.split(), result)

In [ ]:
translate("Can I have some coffee?", model, encoder_tokenizer, decoder_tokenizer, max_len=30)

In [ ]:
translate("May I help you?", model, encoder_tokenizer, decoder_tokenizer, max_len=30)

In [ ]:
translate("The most powerful man all over the world.", model, encoder_tokenizer, decoder_tokenizer, max_len=30)

x축(소스 문장)과 y축(번역 결과)이 만나는 칸이 밝을수록, 그 단어를 생성할 때
해당 소스 단어에 강하게 집중(attention)했다는 뜻입니다.
`coffee → café` 처럼 대응되는 단어 쌍에서 밝은 칸이 나오면 성공!

---
## 5. [프로젝트] Seq2Seq으로 한국어 번역기 만들기

이제 실습 코드를 활용해 **한국어 → 영어** 번역기를 직접 만들어봅니다.

> 참고: 데이터의 난이도가 높은 편이므로 생각만큼 결과가 잘 안나올 수 있습니다.
> 난이도에 비해 데이터가 많지 않아 **훈련 데이터와 검증 데이터를 따로 나누지는 않습니다.**

### 라이브러리 버전을 확인해 봅니다

In [ ]:
import pandas
import torch
import matplotlib

print(pandas.__version__)
print(torch.__version__)
print(matplotlib.__version__)

### Step 1. 데이터 다운로드

`korean-english-park.train.tar.gz` 를 다운로드받아 한영 병렬 데이터를 확보합니다.

- 출처: [jungyeul/korean-parallel-corpora](https://github.com/jungyeul/korean-parallel-corpora)

In [ ]:
kor_dataset_dir = os.path.join(PROJECT_DIR, "datasets")
tar_path = os.path.join(kor_dataset_dir, "korean-english-park.train.tar.gz")

if not os.path.exists(tar_path):
    print("데이터 다운로드 중...")
    url = ("https://github.com/jungyeul/korean-parallel-corpora/raw/master/"
           "korean-english-news-v1/korean-english-park.train.tar.gz")
    urllib.request.urlretrieve(url, tar_path)
    print("다운로드 완료!")

ko_path = os.path.join(kor_dataset_dir, "korean-english-park.train.ko")
if not os.path.exists(ko_path):
    print("압축 해제 중...")
    with tarfile.open(tar_path, "r:gz") as tar_ref:
        tar_ref.extractall(kor_dataset_dir)
    print("압축 해제 완료!")

en_path = os.path.join(kor_dataset_dir, "korean-english-park.train.en")

with open(ko_path, "r", encoding="utf-8") as f:
    kor_raw = f.read().splitlines()
with open(en_path, "r", encoding="utf-8") as f:
    eng_raw = f.read().splitlines()

print("한국어 문장 수:", len(kor_raw))
print("영어 문장 수:", len(eng_raw))
print(kor_raw[0])
print(eng_raw[0])

### Step 2. 데이터 정제

1. **`set` 데이터형이 중복을 허용하지 않는다는 것을 활용해 중복된 데이터를 제거**합니다.
   데이터의 병렬 쌍이 흐트러지지 않게 주의하세요! (쌍을 tuple로 묶어서 set에 넣기)
   중복을 제거한 데이터를 `cleaned_corpus` 에 저장합니다.
2. 앞서 정의한 `preprocess_sentence()` 함수는 한글에서는 동작하지 않습니다
   (`[^a-zA-Z?.!,]` 정규식이 한글을 다 지워버려요!).
   **한글에 적용할 수 있는 정규식을 추가하여 함수를 재정의**합니다.
3. 한글 토큰화는 **mecab**을 사용합니다. (KoNLPy의 mecab 클래스 대신,
   같은 엔진을 쓰는 `python-mecab-ko` 패키지 사용 — Mac에서 설치가 훨씬 간단합니다.)
4. 모든 데이터를 사용할 경우 학습에 굉장히 오랜 시간이 걸리므로,
   **토큰 길이가 40 이하인 데이터를 선별**하여 `eng_corpus` 와 `kor_corpus` 를 구축합니다.

In [ ]:
from mecab import MeCab   # python-mecab-ko (KoNLPy Mecab과 같은 형태소 분석 엔진)

mecab = MeCab()
print(mecab.morphs("나는 커피가 필요 없다."))

In [ ]:
# 1. set으로 중복 제거 — (한국어, 영어) 쌍을 통째로 묶어야 병렬 쌍이 흐트러지지 않습니다.
cleaned_corpus = list(set(zip(kor_raw, eng_raw)))
print("중복 제거 전:", len(kor_raw))
print("중복 제거 후:", len(cleaned_corpus))

In [ ]:
# 2. 한글용 전처리 함수 재정의
def preprocess_korean(sentence):
    sentence = sentence.strip()

    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)      # 구두점 앞뒤에 공백
    sentence = re.sub(r'[" "]+', " ", sentence)             # 연속 공백 → 하나
    sentence = re.sub(r"[^가-힣0-9?.!,]+", " ", sentence)   # 한글/숫자/구두점 외 제거

    return sentence.strip()

# 영어는 실습 때 정의한 preprocess_sentence()를 그대로 사용합니다.
print(preprocess_korean("나는 커피가 필요 없다!!! ☕"))
print(preprocess_sentence("I don't need coffee!!!"))

In [ ]:
# 3~4. 전처리 + mecab 토큰화 + 토큰 길이 40 이하 선별
kor_corpus = []
eng_corpus = []

for kor, eng in tqdm(cleaned_corpus):
    kor = preprocess_korean(kor)
    eng = preprocess_sentence(eng)

    if len(kor) == 0 or len(eng) == 0:
        continue

    # 한글은 mecab 형태소 단위로, 영어는 공백 단위로 토큰화해 길이를 확인합니다.
    kor_tokens = mecab.morphs(kor)
    eng_tokens = eng.split()

    if len(kor_tokens) > 40 or len(eng_tokens) > 40:
        continue

    # mecab 토큰을 공백으로 이어 붙여 저장 → SentencePiece 학습이 쉬워집니다.
    kor_corpus.append(" ".join(kor_tokens))
    eng_corpus.append(eng)

print("선별된 문장 쌍:", len(kor_corpus))
print(kor_corpus[100])
print(eng_corpus[100])

### Step 3. 데이터 토큰화

정제된 말뭉치로 tokenizer를 만듭니다. 원문은 "단어 수 최소 10,000 이상"을 요구하므로
**vocab_size = 10000** 으로 SentencePiece를 학습하고, 실습에서 만든
`TranslationDataset` 을 재활용해 텐서로 변환합니다.

> 주의: 난이도에 비해 데이터가 많지 않아 훈련/검증 데이터를 따로 나누지는 않습니다.

In [ ]:
# 말뭉치를 파일로 저장 (SentencePiece 학습용)
pd.Series(kor_corpus).to_csv("kor_corpus.txt", index=False, header=False, sep="\n", encoding="utf-8")
pd.Series(eng_corpus).to_csv("eng_corpus_proj.txt", index=False, header=False, sep="\n", encoding="utf-8")

kor_vocab_size = 10000   # 최소 10,000 이상 (실험을 통해 조정해보세요)

# Encoder(한국어)용 tokenizer
spm.SentencePieceTrainer.train(
    input="kor_corpus.txt",
    model_prefix="kor_spm",
    vocab_size=kor_vocab_size,
    pad_id=pad_id, bos_id=bos_id, eos_id=eos_id, unk_id=unk_id
)

# Decoder(영어)용 tokenizer
spm.SentencePieceTrainer.train(
    input="eng_corpus_proj.txt",
    model_prefix="eng_spm",
    vocab_size=kor_vocab_size,
    pad_id=pad_id, bos_id=bos_id, eos_id=eos_id, unk_id=unk_id
)

kor_tokenizer = spm.SentencePieceProcessor()
kor_tokenizer.load("kor_spm.model")

eng_tokenizer = spm.SentencePieceProcessor()
eng_tokenizer.load("eng_spm.model")

print("한국어 vocab:", len(kor_tokenizer), "/ 영어 vocab:", len(eng_tokenizer))

In [ ]:
# TranslationDataset은 'eng'/'spa' 컬럼명을 사용하므로 같은 형태의 DataFrame으로 만듭니다.
# (eng 컬럼 = 소스(한국어), spa 컬럼 = 타겟(영어) 역할)
proj_df = pd.DataFrame({"eng": kor_corpus, "spa": eng_corpus})

PROJ_MAX_LEN = 40    # 토큰 길이 40 이하로 선별했으므로
PROJ_BATCH_SIZE = 64

proj_dataset = TranslationDataset(proj_df, kor_tokenizer, eng_tokenizer, max_len=PROJ_MAX_LEN)
proj_loader = DataLoader(proj_dataset, batch_size=PROJ_BATCH_SIZE, shuffle=True)

for src, trg_input, trg_label in proj_loader:
    print(src.shape, trg_input.shape, trg_label.shape)
    break

### Step 4. 모델 설계

한국어를 영어로 잘 번역해 줄 멋진 **Attention 기반 Seq2seq 모델**을 설계하세요!
`Embedding Size` 와 `Hidden Size` 는 실험을 통해 적당한 값을 맞춰 주도록 합니다.
(아래는 실습과 같은 256/512로 시작 — 더 키우거나 줄여보세요.)

In [ ]:
proj_input_dim = len(kor_tokenizer)
proj_output_dim = len(eng_tokenizer)
proj_emb_dim = 256   # Embedding Size — 실험해보세요
proj_hid_dim = 512   # Hidden Size — 실험해보세요

proj_encoder = Encoder(proj_input_dim, proj_emb_dim, proj_hid_dim).to(device)
proj_attention = BahdanauAttention(proj_hid_dim).to(device)
proj_decoder = Decoder(proj_output_dim, proj_emb_dim, proj_hid_dim, proj_attention).to(device)
proj_model = Seq2SeqAttention(proj_encoder, proj_decoder, device).to(device)

print(proj_model)

### Step 5. 훈련하기

훈련엔 위에서 사용한 코드를 그대로 사용하되, **`eval_step()` 부분이 없음**에 유의합니다!
(훈련/검증 데이터를 나누지 않았으니까요.)

> 참고: 데이터의 난이도가 높은 편이므로 생각만큼 결과가 잘 안나올 수 있습니다.
> Mac(mps)에서는 데이터가 실습보다 많아 시간이 꽤 걸립니다 — EPOCHS를 작게 시작해보세요.

In [ ]:
proj_optimizer = optim.Adam(proj_model.parameters(), lr=1e-3)
proj_criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

print("슝~")

In [ ]:
%%time

PROJ_EPOCHS = 10   # 시간을 보며 조절하세요

for epoch in range(PROJ_EPOCHS):
    train_loss = train_step(proj_model, proj_loader, proj_optimizer, proj_criterion, epoch)
    print(f'Epoch {epoch+1}/{PROJ_EPOCHS}, Train Loss: {train_loss:.4f}')

매 스텝 아래의 예문에 대한 번역을 생성하여, 본인이 생각하기에 **가장 멋지게 번역한
Case를 제출**하세요! (Attention Map을 시각화해보는 것도 재밌을 거예요!)

```
## 예문 ##
K1) 오바마는 대통령이다.
K2) 시민들은 도시 속에 산다.
K3) 커피는 필요 없다.
K4) 일곱 명의 사망자가 발생했다.

## 제출 (예시) ##
E1) obama is the president . <end>
E2) people are victims of the city . <end>
E3) the price is not enough . <end>
E4) seven people have died . <end>
```

한국어 문장도 **훈련 때와 동일하게** 전처리(`preprocess_korean` + mecab 토큰화)해서
넣어야 한다는 점에 주의하세요.

In [ ]:
def translate_korean(sentence, model, kor_tokenizer, eng_tokenizer, max_len=40):
    """한국어 문장을 영어로 번역하고 Attention Map을 그립니다."""
    model.eval()

    # 훈련 데이터와 동일한 전처리: 정제 → mecab 형태소 분리 → 공백 join
    cleaned = preprocess_korean(sentence)
    tokenized = " ".join(mecab.morphs(cleaned))

    src_ids = kor_tokenizer.encode(tokenized)
    src_ids = src_ids[:max_len]
    src_ids = src_ids + [0] * (max_len - len(src_ids))   # 패딩 추가
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)

    with torch.no_grad():
        outputs, attentions = model(src_tensor, max_len=max_len)

    result = [eng_tokenizer.decode([token.item()]) for token in outputs.argmax(2).squeeze(1)]
    if "<end>" in result:
        result = result[:result.index("<end>")]

    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

    attention = attentions.squeeze(1).cpu().numpy()
    attention = attention[:len(result), :len(tokenized.split())]
    plot_attention(attention, tokenized.split(), result)

In [ ]:
translate_korean("오바마는 대통령이다.", proj_model, kor_tokenizer, eng_tokenizer)

In [ ]:
translate_korean("시민들은 도시 속에 산다.", proj_model, kor_tokenizer, eng_tokenizer)

In [ ]:
translate_korean("커피는 필요 없다.", proj_model, kor_tokenizer, eng_tokenizer)

In [ ]:
translate_korean("일곱 명의 사망자가 발생했다.", proj_model, kor_tokenizer, eng_tokenizer)

---
## 마무리

이번 프로젝트에서 한 것들:

1. **데이터 전처리** — 정제(정규식) → 토큰화(SentencePiece / mecab) → 패딩 → DataLoader
2. **모델 설계** — GRU Encoder-Decoder + **Bahdanau Attention** (W1, W2, v로 additive scoring)
3. **훈련** — Teacher Forcing 기반 `train_step`, `<PAD>` mask(`ignore_index`),
   gradient clipping, `eval_step`으로 과적합 감시
4. **평가** — 추론 모드(자기 예측을 다음 입력으로) 번역 + **Attention Map 시각화**

**더 실험해볼 것들**
- `EPOCHS`, `Embedding/Hidden Size`, `vocab_size` 바꿔가며 번역 품질 비교
- 예문 4개의 번역 결과 중 가장 멋진 Case를 골라 프로젝트 제출!